# 13 · Stable Diffusion Inpainting (검은 마스크 → 진짜 변형된 얼굴)

**목적**: SC-FEGAN의 TF 1.x 환경 제약을 우회. 동일 task(mask-based facial inpainting)를 **Stable Diffusion Inpaint (Rombach et al. 2022)**로 수행.

**파이프라인**:
```
사진 → 랜드마크 → 시술별 mask → 시술별 prompt → SD Inpaint → Refinement → 최종
```

**환경**: Colab T4 GPU + PyTorch + diffusers (Python 3.12 안정 호환)

**산출물**: `/MyDrive/cv-final-ckpts/samples/sd_inpaint_{procedure}.png`

**학술 framing**: SC-FEGAN(2019)은 TF 1.15 환경 의존성으로 Python 3.12에서 inference 불가. 동일 task를 더 최신 모델(SD Inpaint)로 대체하여 본 프로젝트의 **모듈식 설계**(시술 추천, 자동 입력 생성, refinement)의 강점을 검증.

## 1. 환경 셋업

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'clone 실패: {result.stderr}')
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    
    print('=== 의존성 설치 ===')
    !pip install -q diffusers transformers accelerate scipy mediapipe opencv-python pyyaml segmentation_models_pytorch
    
    import torch
    print(f'\n✅ PyTorch {torch.__version__}')
    print(f'✅ CUDA: {torch.cuda.is_available()}')
    print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Stable Diffusion Inpainting 모델 로드

**모델**: `stabilityai/stable-diffusion-2-inpainting` (5GB, 첫 다운로드 2-5분)

T4 GPU에서 float16으로 메모리 최적화.

In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler

MODEL_ID = 'stabilityai/stable-diffusion-2-inpainting'

print(f'모델 로드 중: {MODEL_ID}')
print('(첫 실행 시 5GB 다운로드 — 2-5분 소요)')

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    revision='fp16',
    safety_checker=None,  # 의료/시술 시뮬레이션이라 NSFW 필터 우회
    requires_safety_checker=False,
).to('cuda')

# DPMSolver — 빠른 sampling (20 steps로도 좋은 결과)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Memory optimization
pipe.enable_attention_slicing()
print('✅ SD Inpainting 파이프라인 준비 완료')

## 3. 사진 + 랜드마크 추출

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

SIZE = 512
OUTPUT_DIR = Path('outputs/sd_inpaint')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드 (정면, 영문 파일명 권장):')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'demo_face.jpg'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

import mediapipe as mp
with mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True,
) as fm:
    res = fm.process(image_rgb)

if not res.multi_face_landmarks:
    raise RuntimeError('얼굴 감지 실패')

lms = res.multi_face_landmarks[0].landmark
landmarks = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in lms], dtype=np.int32)
print(f'✅ Landmarks: {landmarks.shape}')

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input')
plt.show()

## 4. 시술별 mask + prompt 설정

In [ ]:
from project.input_generator import make_mask, load_procedures

procedures_db = load_procedures()

# 시술별 SD 프롬프트 (자연스러운 결과 위해 정교히)
PROMPTS = {
    'nose_tip': {
        'positive': 'refined and slightly lifted nose tip, slim nose bridge, natural Asian face, photorealistic, detailed skin texture, professional photography, soft natural lighting',
        'negative': 'deformed nose, ugly, blurry, low quality, distorted face, cartoon, anime, oversaturated, makeup',
    },
    'double_eyelid': {
        'positive': 'natural double eyelid, subtle eye crease, bright open eyes, natural Asian beauty, photorealistic, detailed eyes, soft skin',
        'negative': 'closed eyes, distorted eyes, ugly, blurry, makeup heavy, cartoon, anime, asymmetric',
    },
    'jaw_v_line': {
        'positive': 'slim v-line jaw, refined chin, natural Asian face, photorealistic, soft jawline, smooth skin, professional photography',
        'negative': 'square jaw, wide face, deformed, ugly, blurry, distorted, cartoon, anime',
    },
}

# 시술별 mask 생성 (이미 작성한 input_generator 활용)
masks = {}
for proc_id in PROMPTS.keys():
    proc = procedures_db[proc_id]
    mask_arr = make_mask(landmarks, proc, size=SIZE)  # (H, W, 1)
    masks[proc_id] = mask_arr[..., 0]  # (H, W)
    print(f'{proc_id}: mask 픽셀 {(mask_arr > 0).sum()}, prompt: "{PROMPTS[proc_id]["positive"][:60]}..."')

## 5. Stable Diffusion Inpainting 실행

In [ ]:
import torch

# PIL 변환 헬퍼
def np_to_pil(arr):
    return Image.fromarray(arr)

image_pil = np_to_pil(image_rgb)

results_sd = {}
torch.manual_seed(42)

for proc_id, prompt_info in PROMPTS.items():
    print(f'\n=== {proc_id} inpainting (~30초) ===')
    
    mask_pil = np_to_pil(masks[proc_id])
    
    result = pipe(
        prompt=prompt_info['positive'],
        negative_prompt=prompt_info['negative'],
        image=image_pil,
        mask_image=mask_pil,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=torch.manual_seed(42),
    ).images[0]
    
    results_sd[proc_id] = np.array(result)
    print(f'  ✅ shape: {results_sd[proc_id].shape}')

## 6. Refinement 적용

In [ ]:
import torch
from project.refinement.model import build_refinement_wrapper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_CKPT = '/content/drive/MyDrive/cv-final-ckpts/refinement_best.pt'

if os.path.exists(REF_CKPT):
    ref_model = build_refinement_wrapper(encoder_weights=None).to(device)
    ref_model.load_state_dict(torch.load(REF_CKPT, map_location=device))
    ref_model.eval()
    print(f'✅ Refinement 로드: {REF_CKPT}')
else:
    ref_model = None
    print('⚠ Refinement ckpt 없음 → SD 결과 그대로 사용')

def refine(img_rgb):
    if ref_model is None:
        return img_rgb
    tensor = torch.from_numpy(img_rgb).float() / 127.5 - 1.0
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = ref_model(tensor)[0].cpu()
    return ((out.clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).numpy()

results_final = {}
for proc_id, sd_out in results_sd.items():
    results_final[proc_id] = refine(sd_out)
    print(f'  {proc_id}: refined')

## 7. 결과 시각화 (4컬럼 비교)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for i, proc_id in enumerate(PROMPTS.keys()):
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'{proc_id}\nBefore (원본)')
    axes[i, 1].imshow(masks[proc_id], cmap='gray')
    axes[i, 1].set_title('Mask\n(시술 영역)')
    axes[i, 2].imshow(results_sd[proc_id])
    axes[i, 2].set_title('SD Inpaint ⭐\n(GAN 결과)')
    axes[i, 3].imshow(results_final[proc_id])
    axes[i, 3].set_title('+ Refinement\n(최종)')
    for ax in axes[i]:
        ax.axis('off')
    
    cv2.imwrite(
        str(OUTPUT_DIR / f'sd_final_{proc_id}.png'),
        cv2.cvtColor(results_final[proc_id], cv2.COLOR_RGB2BGR),
    )
    cv2.imwrite(
        str(OUTPUT_DIR / f'sd_only_{proc_id}.png'),
        cv2.cvtColor(results_sd[proc_id], cv2.COLOR_RGB2BGR),
    )

plt.tight_layout()
save_path = OUTPUT_DIR / 'all_sd_results.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_path}')

## 8. Drive 백업

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -r {OUTPUT_DIR}/* {DRIVE_OUT}/
    print(f'✅ Drive 백업: {DRIVE_OUT}/')
    !ls -lh {DRIVE_OUT}/sd_*.png 2>/dev/null

print('\n=== Phase 7-G 완료 ===')
print('산출물 활용:')
print('  1. 발표 슬라이드 — sd_final_*.png 4컬럼 비교 이미지')
print('  2. Streamlit — 정적 이미지로 표시 (실시간 inference X)')
print('  3. PDF 견적서 — Before/After 영역에 활용')

---

## 발표 framing (학술 정직성)

**기술 선택 정당화**:
1. 원 계획: SC-FEGAN (Jo & Park 2019, ICCV)
2. 문제: TF 1.15 의존, Python 3.12 환경에서 inference 불가
3. 우회: **동일 task(mask-based facial inpainting)를 수행하는 최신 모델 SD Inpaint (Rombach et al. 2022, CVPR)** 채택
4. 장점:
   - 더 최신 기술 (2022 vs 2019)
   - 환경 안정성 (PyTorch 기반)
   - 우리 모듈식 설계의 강점 검증 (시술 추천, 자동 입력 생성, Refinement는 model-agnostic)

**참고문헌**:
- Rombach, R. et al. (2022). High-Resolution Image Synthesis with Latent Diffusion Models. CVPR.
- Jo, Y., & Park, J. (2019). SC-FEGAN: Face Editing GAN with User's Sketch and Color. ICCV.